In [1]:
from dataclasses import replace

import tabulate
import pandas as pd

from config.experiment import ExperimentConfig
from config.task import generate_task_configs
from experiments import load_config
from utils.plot import Metric, PlotFilter, get_metrics, plot_metrics_vs_perturbation, shorten_model_name, strip_hf_org, cleanup_colnames_after_groupby, combine_mean_std_columns

/Users/ezaki/Library/CloudStorage/OneDrive-TheAlanTuringInstitute/Research/LLMSec/code/llm_venv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def get_config(cfg_name: str) -> ExperimentConfig:
    exp_cfg = load_config(cfg_name)

    return ExperimentConfig(
        train=replace(exp_cfg.train, save_folder=exp_cfg.train.save_folder.parent / "paper-multirun"),
        eval=replace(exp_cfg.eval, results_folder=exp_cfg.eval.results_folder.parent / "paper-multirun"),
    )

In [4]:
# exp_config = get_config("paper_perturbation_plot")

# fig = plot_metrics_vs_perturbation(
#     exp_config,
#     metrics_to_show=[
#         Metric.RETRIEVAL_ASR_TRAIN,
#         Metric.RETRIEVAL_ASR_TEST,
#         Metric.GENERATION_ASR_EXACT_TRAIN,
#         Metric.GENERATION_ASR_EXACT_TEST,
#         Metric.GENERATION_ACC_EMBED_GT_TRAIN,
#         Metric.GENERATION_ACC_EMBED_GT_TEST,
#     ],
#     ret_topk_idx=0,
#     gen_topk_idx=0,
# )

# from config import OUTPUTS_FOLDER
# fig.savefig(OUTPUTS_FOLDER / "perturbation.pdf", bbox_inches='tight')

In [ ]:
# exp_config = get_config("perturbation_plot_targeted")
#
# plot_metrics_vs_perturbation(
#     exp_config,
#     metrics_to_show=[
#         Metric.RETRIEVAL_ASR_TRAIN,
#         Metric.RETRIEVAL_ASR_TEST,
#         Metric.GENERATION_ASR_EXACT_TRAIN,
#         Metric.GENERATION_ASR_EXACT_TEST,
#         Metric.GENERATION_ACC_EMBED_GT_TRAIN,
#         Metric.GENERATION_ACC_EMBED_GT_TEST,
#     ],
#     ret_topk_idx=0,
#     gen_topk_idx=0,
# )

In [ ]:
tbl_format = "html" # html or latex 
do_combine_aggs = True
do_aggr = True

def make_all_metric_table(config_name: str, metrics_to_show: list[Metric] | None = None, row_filter=None):
    exp_config = get_config(config_name)
    if metrics_to_show is None:
        metrics_to_show = [m for m in Metric]

    task_configs = generate_task_configs(exp_config, include_eval=True)

    table = []
    for task_config in task_configs:
        row = {
            "dataset": strip_hf_org(task_config.ds_name),
        }
        row["image index"] = task_config.chosen_index
        if not exp_config.eval.test_gpt_attack:
            row["embedder"] = shorten_model_name(task_config.model_name_embs[0]) if len(task_config.model_name_embs) == 1 else "+".join([shorten_model_name(m) for m in task_config.model_name_embs])
            if task_config.vlm:
                row["vlm"] = shorten_model_name(task_config.vlm.models[0]) if len(task_config.vlm.models) == 1 else "+".join([shorten_model_name(m) for m in task_config.vlm.models])
                if len(exp_config.train.vlm.gen_topk_list) > 1:
                    row["vlm topk"] = task_config.vlm.gen_topk
        if len(exp_config.train.emb_train_loss_type_list) > 1:
            row["emb train loss"] = task_config.emb_train_loss_type
        if len(exp_config.train.attack_mask_list) > 1:
            row["attack mask"] = task_config.attack_mask.name
        if task_config.eval_emb_name:
            row["eval emb"] = shorten_model_name(task_config.eval_emb_name)
        if task_config.eval_vlm_name:
            row["eval vlm"] = shorten_model_name(task_config.eval_vlm_name)

        if task_config.judge:
            row["judge"] = shorten_model_name(task_config.judge.model_name)
        if task_config.eval_jdg_name:
            row["eval judge"] = shorten_model_name(task_config.eval_jdg_name)
        metrics, _ = get_metrics(
            exp_config=exp_config,
            task_config=task_config,
            metrics_to_show=metrics_to_show,
        )
        row.update(metrics)
        table.append(row)
    if row_filter:
        table = [row for row in table if row_filter(row)]
    
    tabulate_table = tabulate.tabulate(table, headers="keys", tablefmt=tbl_format, showindex="never")

    # aggreggate similar settings and show mean and std
    if do_aggr:
        df = pd.DataFrame(table)
        
        grouping_columns = ['dataset', 'embedder', 'vlm', 'eval emb', 'eval vlm']
        excluded_columns = grouping_columns + ['image index']
        aggregate_columns = [col for col in df.columns.tolist() if col not in excluded_columns]

        agg_dict = {col: ['mean', 'std'] for col in aggregate_columns}
        agg_df = df.groupby(grouping_columns).agg(agg_dict).reset_index()
        agg_df.columns = cleanup_colnames_after_groupby(agg_df.columns)
        
        if do_combine_aggs: combine_mean_std_columns(agg_df, aggregate_columns)
        
        tabulate_table = tabulate.tabulate(agg_df, headers="keys", tablefmt=tbl_format, showindex="never")
    
    return tabulate_table

In [7]:
make_all_metric_table("paper_non_targeted", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED, row_filter=PlotFilter.CONDITION_SAME_MODELS)

dataset,embedder,vlm,eval emb,eval vlm,Recall-B@1 (mean ± std),Recall-A@1 (mean ± std),ASR-R (test)@1 (mean ± std),Recall-B@5 (mean ± std),Recall-A@5 (mean ± std),ASR-R (test)@5 (mean ± std),ASR-G-HARD (test)@-1 (mean ± std),SIM-G-ADV (test)@-1 (mean ± std),SIM-G-GT (test)@-1 (mean ± std)
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,CLIP-L,InternVL3-2B,0.21 ± 0.00,0.02 ± 0.01,0.97 ± 0.03,0.44 ± 0.00,0.43 ± 0.00,1.00 ± 0.00,0.96 ± 0.07,0.96 ± 0.07,0.04 ± 0.03
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,CLIP-L,Qwen2.5-VL-3B,0.21 ± 0.00,0.02 ± 0.01,0.98 ± 0.03,0.44 ± 0.00,0.43 ± 0.00,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00,0.03 ± 0.00
syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,CLIP-L,SmolVLM,0.21 ± 0.00,0.04 ± 0.03,0.90 ± 0.14,0.44 ± 0.00,0.43 ± 0.00,0.99 ± 0.02,1.00 ± 0.00,1.00 ± 0.00,0.03 ± 0.00
syntheticDocQA_artificial_intelligence_test,ColPali,InternVL3-2B,ColPali,InternVL3-2B,0.00 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.01 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.49 ± 0.38,0.51 ± 0.39,0.25 ± 0.21
syntheticDocQA_artificial_intelligence_test,ColPali,Qwen2.5-VL-3B,ColPali,Qwen2.5-VL-3B,0.00 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.01 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00,1.00 ± 0.00,0.03 ± 0.00
syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,ColPali,SmolVLM,0.00 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.01 ± 0.00,0.00 ± 0.00,1.00 ± 0.00,0.57 ± 0.50,0.60 ± 0.49,0.20 ± 0.21
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,InternVL3-2B,GME-Qwen2-VL-2B,InternVL3-2B,0.58 ± 0.00,0.58 ± 0.01,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.01,0.19 ± 0.13,1.00 ± 0.00,1.00 ± 0.00,0.03 ± 0.00
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.58 ± 0.00,0.58 ± 0.00,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.01,0.17 ± 0.11,1.00 ± 0.00,1.00 ± 0.00,0.03 ± 0.00
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,SmolVLM,GME-Qwen2-VL-2B,SmolVLM,0.58 ± 0.00,0.58 ± 0.00,0.00 ± 0.00,0.94 ± 0.00,0.94 ± 0.01,0.13 ± 0.10,0.99 ± 0.02,0.99 ± 0.02,0.03 ± 0.01


In [ ]:
make_all_metric_table("paper_targeted_attacks_oneQ_oneA", row_filter=PlotFilter.CONDITION_SAME_MODELS, metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

"('dataset', '')","('embedder', '')","('vlm', '')","('eval emb', '')","('eval vlm', '')","('ASR-R targeted@1', 'mean')","('ASR-R targeted@1', 'std')","('FPR-R targeted (test)@1', 'mean')","('FPR-R targeted (test)@1', 'std')","('ASR-R targeted@5', 'mean')","('ASR-R targeted@5', 'std')","('FPR-R targeted (test)@5', 'mean')","('FPR-R targeted (test)@5', 'std')","('SIM-G-ADV-POS targeted (train)@-1', 'mean')","('SIM-G-ADV-POS targeted (train)@-1', 'std')","('SIM-G-ADV-NEG targeted (test)@-1', 'mean')","('SIM-G-ADV-NEG targeted (test)@-1', 'std')"
syntheticDocQA_artificial_intelligence_test,CLIP-L,InternVL3-2B,CLIP-L,InternVL3-2B,1,0,0,0,1,0,0.03,0.0273861,0.995096,0.00797263,0.214422,0.0185118
syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,CLIP-L,Qwen2.5-VL-3B,1,0,0,0,1,0,0.01,0.0223607,0.886089,0.254712,0.215629,0.0151147
syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,CLIP-L,SmolVLM,1,0,0,0,1,0,0.01,0.0223607,0.979908,0.0449258,0.228498,0.0271477
syntheticDocQA_artificial_intelligence_test,ColPali,InternVL3-2B,ColPali,InternVL3-2B,0.6,0.547723,0,0,0.6,0.547723,0,0,0.748727,0.222518,0.213753,0.0164692
syntheticDocQA_artificial_intelligence_test,ColPali,Qwen2.5-VL-3B,ColPali,Qwen2.5-VL-3B,0.6,0.547723,0,0,0.8,0.447214,0,0,0.908651,0.192632,0.215839,0.0186658
syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,ColPali,SmolVLM,0.6,0.547723,0,0,1,0,0,0,0.718385,0.25731,0.213957,0.0165288
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,InternVL3-2B,GME-Qwen2-VL-2B,InternVL3-2B,0.8,0.447214,0,0,1,0,0,0,0.974166,0.0577667,0.219459,0.0202113
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.6,0.547723,0,0,1,0,0,0,1,0,0.211406,0.0149462
syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,SmolVLM,GME-Qwen2-VL-2B,SmolVLM,0.8,0.447214,0,0,1,0,0.01,0.0223607,0.988449,0.0258283,0.218104,0.0119838


In [ ]:
make_all_metric_table("paper_targeted_attacks_multiQ_oneA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

,dataset,embedder,vlm,ASR-R targeted@1,FPR-R targeted (test)@1,ASR-R targeted@5,FPR-R targeted (test)@5,SIM-G-ADV-POS targeted (train)@-1,SIM-G-ADV-NEG targeted (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,1,0,1,0,1,0.174183
1,syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,0.8,0,0.8,0,1,0.492558
2,syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,1,0,1,0,0.0982995,0.00474182
3,syntheticDocQA_artificial_intelligence_test,ColPali,Qwen2.5-VL-3B,0.4,0,1,0,1,0.411617
4,syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,SmolVLM,0.6,0,0.6,0,1,0.0278113
5,syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.6,0,0.8,0,0.792979,0.634126


In [ ]:
make_all_metric_table("paper_targeted_attacks_multiQ_multiA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

,dataset,embedder,vlm,ASR-R targeted@1,FPR-R targeted (test)@1,ASR-R targeted@5,FPR-R targeted (test)@5,SIM-G-ADV-POS targeted (train)@-1,SIM-G-ADV-NEG targeted (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,1,0,1,0,0.999991,0.269673
1,syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,1,0,1,0,0.99998,0.269854
2,syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,1,0,1,0,0.527364,0.260008
3,syntheticDocQA_artificial_intelligence_test,ColPali,Qwen2.5-VL-3B,1,0,1,0,0.907934,0.274945
4,syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,SmolVLM,0.5,0,1,0,0.999989,0.262672
5,syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.5,0,1,0,0.998746,0.257119


In [ ]:
make_all_metric_table("paper_judge_defence", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

,dataset,embedder,vlm,eval judge,Judge Image Content Relevancy (test)@-1,Judge Image Faithfulness (test)@-1,Judge Answer Relevancy (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,SmolVLM,0.55,0.3,0
1,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,Qwen2.5-VL-3B,0,0,0


In [ ]:
make_all_metric_table("paper_judge_defence_adapt", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

,dataset,embedder,vlm,judge,eval judge,Judge Image Content Relevancy (test)@-1,Judge Image Faithfulness (test)@-1,Judge Answer Relevancy (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,SmolVLM,SmolVLM,1,1,1
1,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,SmolVLM,Qwen2.5-VL-3B,0,0,0
2,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,Qwen2.5-VL-3B,SmolVLM,0.45,0.05,0.05
3,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,Qwen2.5-VL-3B,Qwen2.5-VL-3B,1,1,1


In [ ]:
make_all_metric_table("paper_judge_defence_targeted", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

,dataset,embedder,vlm,eval judge,Judge Image Content Relevancy (test)@-1,Judge Image Faithfulness (test)@-1,Judge Answer Relevancy (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,SmolVLM,0.65,0.75,0.2
1,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,Qwen2.5-VL-3B,0,0,0


In [ ]:
make_all_metric_table("paper_judge_defence_targeted_adapt", metrics_to_show=PlotFilter.METRICS_TEST_JUDGE)

,dataset,embedder,vlm,judge,eval judge,Judge Image Content Relevancy (test)@-1,Judge Image Faithfulness (test)@-1,Judge Answer Relevancy (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,SmolVLM,SmolVLM,1,1,1
1,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,SmolVLM,Qwen2.5-VL-3B,0.05,0,0
2,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,Qwen2.5-VL-3B,SmolVLM,0.75,0.65,0
3,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,Qwen2.5-VL-3B,Qwen2.5-VL-3B,1,1,1


In [8]:
make_all_metric_table("paper_copali_ab", metrics_to_show=PlotFilter.METRICS_COLPALI)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/ezaki/Library/CloudStorage/OneDrive-TheAlanTuringInstitute/Research/LLMSec/code/mumorag_gh_2/mumoRAG-attacks/data/results/paper-multirun/metrics_paper_copali_ab_acd67059b8b77e6463b6ecab3605d2ba.json'

In [ ]:
make_all_metric_table("paper_copali_ab_cpoiT", metrics_to_show=PlotFilter.METRICS_COLPALI)

,dataset,embedder,vlm,emb train loss,ASR-R (test)+maxsim@1,ASR-R (test)+maxsim@5,ASR-R (test)+avgsim@1,ASR-R (test)+avgsim@5,ASR-R (test)+softmaxsim@1,ASR-R (test)+softmaxsim@5,ASR-R (test)+cos_avgemb@1,ASR-R (test)+cos_avgemb@5
0,syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,maxsim,0,0.15,0,0,0,0,0,0
1,syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,avgsim,0,0,0.25,0.45,0.15,0.35,0.1,0.2
2,syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,softmaxsim,0,0.05,0.15,0.4,0.05,0.35,0.05,0.2
3,syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,cos_avgemb,0,0.1,0.05,0.05,0,0.05,0.05,0.1


In [ ]:
make_all_metric_table("paper_topk_context", metrics_to_show=PlotFilter.METRICS_TOPK)

,dataset,embedder,vlm,vlm topk,ASR-G-HARD (test)@-1,SIM-G-ADV (test)@-1,SIM-G-GT (test)@-1,ASR-G-HARD (test)@1,SIM-G-ADV (test)@1,SIM-G-GT (test)@1,ASR-G-HARD (test)@5,SIM-G-ADV (test)@5,SIM-G-GT (test)@5
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,1,1,1,0.0301554,0.95,0.947907,0.0650789,0,-0.0287684,0.558359
1,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,5,1,1,0.0301554,0.95,0.946058,0.066424,1,1,0.0301554


In [ ]:
make_all_metric_table("paper_topk_context_targeted", metrics_to_show=PlotFilter.METRICS_TOPK_TARGETED)

,dataset,embedder,vlm,vlm topk,SIM-G-ADV-POS targeted (train)@-1,SIM-G-ADV-NEG targeted (test)@-1,SIM-G-ADV-POS targeted (train)@1,SIM-G-ADV-NEG targeted (test)@1,SIM-G-ADV-POS targeted (train)@5,SIM-G-ADV-NEG targeted (test)@5
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,1,1,-0.00262511,1,-0.0140627,-0.0648307,-0.025339
1,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,5,1,0.0214569,1,-0.0181628,1,-0.0268381


In [ ]:
make_all_metric_table("paper_defences", metrics_to_show=PlotFilter.METRICS_TOPK)

,dataset,embedder,vlm,ASR-G-HARD (test)@-1,SIM-G-ADV (test)@-1,SIM-G-GT (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,0.95,0.961232,0.061236


In [ ]:
make_all_metric_table("paper_targeted_defences", metrics_to_show=PlotFilter.METRICS_TOPK_TARGETED)

,dataset,embedder,vlm,SIM-G-ADV-POS targeted (train)@-1,SIM-G-ADV-NEG targeted (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,1,-0.00303376


In [ ]:
# make_all_metric_table("mask_attack")

In [ ]:
make_all_metric_table("paper_GPT_non_targeted", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED)

,dataset,eval emb,eval vlm,Recall-B@1,Recall-A@1,ASR-R (test)@1,Recall-B@5,Recall-A@5,ASR-R (test)@5,ASR-G-HARD (test)@-1,SIM-G-ADV (test)@-1,SIM-G-GT (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,0.21,0.21,0,0.44,0.44,0,0,-0.0240033,0.549896
1,syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,0.21,0.21,0,0.44,0.44,0,0,0.0144284,0.528623
2,syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,0.66,0.65,0,0.98,0.98,0,0,-0.00428852,0.515134
3,syntheticDocQA_artificial_intelligence_test,ColPali,Qwen2.5-VL-3B,0.66,0.65,0,0.98,0.98,0,0,0.00512495,0.544008
4,syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,SmolVLM,0.58,0.58,0,0.94,0.94,0,0,-0.0230045,0.533048
5,syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.58,0.58,0,0.94,0.94,0,0,0.0189598,0.51165


In [ ]:
make_all_metric_table("paper_GPT_targeted_attacks_oneQ_oneA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

,dataset,eval emb,eval vlm,ASR-R targeted@1,FPR-R targeted (test)@1,ASR-R targeted@5,FPR-R targeted (test)@5,SIM-G-ADV-POS targeted (train)@-1,SIM-G-ADV-NEG targeted (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,1,0,1,0,0.793339,0.271321
1,syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,1,0,1,0,0.89461,0.222097
2,syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,1,0,1,0,0.999978,0.341364
3,syntheticDocQA_artificial_intelligence_test,ColPali,Qwen2.5-VL-3B,1,0,1,0,0.9657,0.23446
4,syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,SmolVLM,1,0,1,0,0.966195,0.303003
5,syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,1,0,1,0,0.928977,0.245825


In [ ]:
make_all_metric_table("paper_GPT_targeted_attacks_multiQ_oneA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

,dataset,eval emb,eval vlm,ASR-R targeted@1,FPR-R targeted (test)@1,ASR-R targeted@5,FPR-R targeted (test)@5,SIM-G-ADV-POS targeted (train)@-1,SIM-G-ADV-NEG targeted (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,0.4,0,0.6,0,-0.0159414,0.0155065
1,syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,0.4,0,0.6,0,-0.0764076,0.011566
2,syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,0.4,0,0.4,0,-0.0225051,-0.0148292
3,syntheticDocQA_artificial_intelligence_test,ColPali,Qwen2.5-VL-3B,0.4,0,0.4,0,-0.11505,0.109386
4,syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,SmolVLM,0.2,0,0.4,0,0.00370789,-0.025899
5,syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.2,0,0.4,0,-0.0463881,-0.0350825


In [ ]:
make_all_metric_table("paper_GPT_targeted_attacks_multiQ_multiA", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

,dataset,eval emb,eval vlm,ASR-R targeted@1,FPR-R targeted (test)@1,ASR-R targeted@5,FPR-R targeted (test)@5,SIM-G-ADV-POS targeted (train)@-1,SIM-G-ADV-NEG targeted (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L,SmolVLM,0.5,0,0.5,0,0.670754,0.320573
1,syntheticDocQA_artificial_intelligence_test,CLIP-L,Qwen2.5-VL-3B,0.5,0,0.5,0,0.87528,0.300047
2,syntheticDocQA_artificial_intelligence_test,ColPali,SmolVLM,0,0,1,0,0.728126,0.312731
3,syntheticDocQA_artificial_intelligence_test,ColPali,Qwen2.5-VL-3B,0,0,1,0,0.884722,0.305826
4,syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,SmolVLM,0.5,0,0.5,0,0.846674,0.309567
5,syntheticDocQA_artificial_intelligence_test,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.5,0,0.5,0,0.830613,0.292673


In [ ]:
make_all_metric_table("paper_multi_transferability", metrics_to_show=PlotFilter.ALL_METRICS_UNTARGETED)

,dataset,embedder,vlm,eval emb,eval vlm,Recall-B@1,Recall-A@1,ASR-R (test)@1,Recall-B@5,Recall-A@5,ASR-R (test)@5,ASR-G-HARD (test)@-1,SIM-G-ADV (test)@-1,SIM-G-GT (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B,CLIP-L,SmolVLM,0.21,0.06,0.9,0.44,0.43,1,0.95,0.951629,0.0464323
1,syntheticDocQA_artificial_intelligence_test,CLIP-L+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B,CLIP-L,Qwen2.5-VL-3B,0.21,0.06,0.9,0.44,0.43,1,1,1,0.0301554
2,syntheticDocQA_artificial_intelligence_test,CLIP-L+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B,GME-Qwen2-VL-2B,SmolVLM,0.58,0.58,0,0.94,0.93,0.25,1,1,0.0301554
3,syntheticDocQA_artificial_intelligence_test,CLIP-L+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0.58,0.58,0,0.94,0.93,0.25,1,1,0.0301554


In [ ]:
make_all_metric_table("paper_multi_transferability_targeted", metrics_to_show=PlotFilter.ALL_METRICS_TARGETED)

,dataset,embedder,vlm,eval emb,eval vlm,ASR-R targeted@1,FPR-R targeted (test)@1,ASR-R targeted@5,FPR-R targeted (test)@5,SIM-G-ADV-POS targeted (train)@-1,SIM-G-ADV-NEG targeted (test)@-1
0,syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B,CLIP-L,SmolVLM,1,0,1,0,1,-0.00830106
1,syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B,CLIP-L,Qwen2.5-VL-3B,1,0,1,0,1,0.157104
2,syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B,ColPali,SmolVLM,1,0,1,0,0.0478707,-0.0188847
3,syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B,ColPali,Qwen2.5-VL-3B,1,0,1,0,1,0.222731
4,syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B,GME-Qwen2-VL-2B,SmolVLM,0,0,1,0,0.0143071,-0.00992858
5,syntheticDocQA_artificial_intelligence_test,CLIP-L+ColPali+GME-Qwen2-VL-2B,SmolVLM+Qwen2.5-VL-3B,GME-Qwen2-VL-2B,Qwen2.5-VL-3B,0,0,1,0,1,0.362108
